In [1]:
import os
import sys

import geopandas as gpd
import yaml

# on se place à la racine du projet
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.append(os.getcwd())

with open("config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

from src.variables import build_part, build_tcam, build_trajectoire

communes = gpd.read_parquet("data/clean/base_finale/communes.parquet")
nb_annees = config["annee_fin"] - config["annee_debut"]

In [2]:
# croissance des voyageurs des gares de la commune (NaN sans gare)
communes["tcam_voyageurs_2017_2023"] = build_tcam(communes["voyageurs_2017"], communes["voyageurs_2023"], nb_annees)

# part des actifs allant travailler en voiture, NaN sous 1 000 actifs
for annee in [config["annee_debut"], config["annee_fin"]]:
    communes[f"part_voiture_{annee}"] = build_part(communes[f"actifs_voiture_{annee}"], communes[f"actifs_{annee}"], config["actifs_min"])

# voitures des particuliers pour 1 000 habitants
communes["voitures_1000_hab_2023"] = communes["voitures_2023"] / communes["pop_2023"].where(communes["pop_2023"] > 0) * 1000

print(communes[["tcam_voyageurs_2017_2023", "part_voiture_2017", "part_voiture_2023", "voitures_1000_hab_2023"]].describe().round(1))

       tcam_voyageurs_2017_2023  part_voiture_2017  part_voiture_2023  \
count                    2435.0             4352.0             4621.0   
mean                        5.3               79.5               79.1   
std                        11.6               12.2               12.8   
min                       -43.0               11.2                9.0   
25%                         0.0               75.8               75.1   
50%                         3.6               83.1               83.2   
75%                         8.1               87.5               87.6   
max                       179.0               95.3               94.9   

       voitures_1000_hab_2023  
count                 34740.0  
mean                    665.7  
std                     167.9  
min                      45.5  
25%                     612.6  
50%                     660.1  
75%                     709.5  
max                   17800.0  
